In [1]:
import configparser
from tqdm.notebook import tqdm
from scipy.ndimage import zoom
import os
import numpy as np
import pylidc as pl
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import inspect
plt.cm.cmap_d = {name: matplotlib.colormaps[name] for name in matplotlib.colormaps}

configparser.SafeConfigParser = configparser.ConfigParser
np.int, np.float, np.bool = int, float, bool

c:\Users\huyge\anaconda3\Lib\site-packages\pylidc\__init__.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as _pr


# Getting to know the data & pylidc library
Dataset used: https://www.cancerimagingarchive.net/collection/lidc-idri/
They advice the use of `pydilc` for the preprocessing and conversions of the data.

**Important!: code on the pydilc site is not up-to-date with the real implementation**

You can print the current implementation of the function `print(inspect.getsource(pl.Scan.###))`
 

In [2]:
print(inspect.getsource(pl.Scan.load_all_dicom_images))

    def load_all_dicom_images(self, verbose=True):
        """
        Load all the DICOM images assocated with this scan and return as list.

        Parameters
        ----------
        verbose: bool
            Turn the loading method on/off.

        Example
        -------
        An example::

            import pylidc as pl
            import matplotlib.pyplot as plt

            scan = pl.query(pl.Scan).first()

            images = scan.load_all_dicom_images()
            zs = [float(img.ImagePositionPatient[2]) for img in images]
            print(zs[1] - zs[0], images[0].SliceThickness, scan.slice_thickness)
            
            plt.imshow(images[0].pixel_array, cmap=plt.cm.gray)
            plt.show()

        """
        if verbose: print("Loading dicom files ... This may take a moment.")

        path = self.get_path_to_dicom_files()
        fnames = [fname for fname in os.listdir(path)
                            if fname.endswith('.dcm') and not fname.startswith(".

In [3]:
print(inspect.getsource(pl.Scan.to_volume))

    def to_volume(self, verbose=True):
        """
        Return the scan as a 3D numpy array volume.
        """
        images = self.load_all_dicom_images(verbose=verbose)

        volume = np.stack(
            [
                x.pixel_array * x.RescaleSlope + x.RescaleIntercept
                for x in images
            ],
            axis=-1,
        ).astype(np.int16)
        return volume



Out of this we can conclude that HU-conversion already happens under the hood in the `pylidc` library when calling `to_volume()` on a scan.

## Loading one patient data

In [9]:
pid = 'LIDC-IDRI-0078'
scan = pl.query(pl.Scan).filter(pl.Scan.patient_id == pid).first() ## Different possible filters (see: https://pylidc.github.io/scan.html#pylidc.Scan). .first() returns the first scan that matches the filter.
print(len(scan.annotations)) ## Show the amount of annotations for this scan
## To cluster the different annotations into nodules, we can use the cluster_annotations() method. This will return a list of lists, where each inner list contains the annotations that belong to the same nodule.
nodules = scan.cluster_annotations()
print("Nodules list:", nodules)
for i,nod in enumerate(nodules):
    print("Nodule %d has %d annotations." % (i+1, len(nodules[i])))

#scan.visualize(annotation_groups=nodules)

# To convert image scan to numpy array, we can use the to_volume() method. 
volume = scan.to_volume(verbose=False)          
print("Volume min:", volume.min())
print("Volume max:", volume.max())
print("Volume shape:", volume.shape) ## The shape of the volume is (height, width, slices)
## So 'slices' amount of CT images of size (height, width) are stacked together to form the volume. The slices are ordered from top to bottom of the body. The pixel values are in Hounsfield units (HU).

13
Nodules list: [[Annotation(id=2,scan_id=1), Annotation(id=6,scan_id=1), Annotation(id=10,scan_id=1), Annotation(id=13,scan_id=1)], [Annotation(id=1,scan_id=1), Annotation(id=5,scan_id=1), Annotation(id=9,scan_id=1), Annotation(id=12,scan_id=1)], [Annotation(id=8,scan_id=1)], [Annotation(id=3,scan_id=1), Annotation(id=4,scan_id=1), Annotation(id=7,scan_id=1), Annotation(id=11,scan_id=1)]]
Nodule 1 has 4 annotations.
Nodule 2 has 4 annotations.
Nodule 3 has 1 annotations.
Nodule 4 has 4 annotations.
Volume min: -2048
Volume max: 8155
Volume shape: (512, 512, 87)


## Annotations

In [17]:
# Getting one annotations
ann = pl.query(pl.Annotation).filter(pl.Annotation.spiculation > 3).first() ## Get the first annotation of the first nodule who has spiculation > 3
ann.print_formatted_feature_table()
## Plot the annotation in 3D and in scan view.
#ann.visualize_in_3d(edgecolor='green', cmap='autumn')
## Plot the annotation in scan view. The scan view shows the CT images of the scan with the annotation overlaid in red on top.
#ann.visualize_in_scan()

Feature              Meaning                    # 
-                    -                          - 
Subtlety           | Obvious                  | 5 
Internalstructure  | Soft Tissue              | 1 
Calcification      | Non-central              | 4 
Sphericity         | Ovoid/Round              | 4 
Margin             | Near Sharp               | 4 
Lobulation         | Near Marked Lobulation   | 4 
Spiculation        | Near Marked Spiculation  | 4 
Texture            | Solid                    | 5 
Malignancy         | Highly Suspicious        | 5 


# Preprocessing of data


### Helperfunctions

## Resampling

This is needed because the voxel size in one image is not the same as in another image.
<br>
For example: in one image it may be: 0,7 × 0,7 × 2,5 mm but in another one: 0,9 × 0,9 × 1,0 mm
<br>
Then the same physical distance in real life wil not be the same distance on the scan.
<br>
We resample everything to 1 × 1 × 1 mm

In [18]:
def resample_scan(volume, scan, target_spacing=(1.0, 1.0, 1.0)):
    # Note: pylidc's volume has the shape (Height, Width, Slices/Depth).
    current_spacing = np.array([float(scan.pixel_spacing), float(scan.pixel_spacing), float(scan.slice_thickness)])
    
    target_spacing = np.array(target_spacing)
    
    # Calculate scaling factor 
    zoom_factors = current_spacing / target_spacing
    
    # order=1 is linear interpolation
    ## TODO: look into other interpolation methods
    volume_resampled = zoom(volume, zoom_factors, order=1)
    
    return volume_resampled, zoom_factors

In [22]:
def process_scan(scan):
    ## Convert the scan to a 3D volume
    vol = scan.to_volume(verbose=False)          # (x, y, z (slices)) already in HU

    # Resampling to voxels of 1 x 1 x 1 mm^3
    volume_resampled, zoom_factors = resample_scan(vol, scan)

    ## Clipping to make the parts of interest more visible. So values of hu_max are a 1 (white) a nd values of hu_min are a 0 (black). The values in between are scaled linearly.
    # Healthy lung tissue has a Hounsfield unit (HU) value of around -950 to -650 (https://radiopaedia.org/articles/hounsfield-unit)
    # However we also wan't to include the nodules which typically have a higher HU value.
    hu_min = -1000
    hu_max = 400
    vol_clipped = np.clip(volume_resampled, hu_min, hu_max).astype(np.float32)
    
    # Min/max normalisation so pixels are between [0,1] in this way we avoid exploding gradients and improve the initial learning (ReLU blocks negative values)
    volume_normalized = (vol_clipped - hu_min) / (hu_max - hu_min)
    
    ## Converting it from (X,Y,Z) to (Z,Y,X), since that is what the model/Conv3D wants to receive
    # Because: batch, channels, depth, height, width
    # Later we add batch and channel dimension with the .unsqueeze function
    processed_volume = np.transpose(volume_normalized, (2, 0, 1)) 
    

    return processed_volume, zoom_factors

The function will cut a 3D-region anround one nodule:

In [ ]:
def extract_patch(volume, center, patch_size=(36, 48, 48)):

    dz, dy, dx = patch_size
    cz, cy, cx = int(center[0]), int(center[1]), int(center[2])

    # Calculate the borders
    original_z_min, original_z_max = cz - (dz // 2), cz + (dz - (dz // 2)) # Note: // 2 is integer devision so it doesn't result in a float
    original_y_min, original_y_max = cy - (dy // 2), cy + (dy - (dy // 2))
    original_x_min, original_x_max = cx - (dx // 2), cx + (dx - (dx // 2))

    # Get volume shape:
    max_z, max_y, max_x = volume.shape

    # Calculate padding needed (needed for new version padding):
    pad_before_z = max(0, -original_z_min) # For example if original_z_min = -13 then there are 13 voxels missing at that side
    pad_after_z = max(0, original_z_max - max_z)

    pad_before_y = max(0, -original_y_min)
    pad_after_y = max(0, original_y_max - max_y)

    pad_before_x = max(0, -original_x_min)
    pad_after_x = max(0, original_x_max - max_x)


    # Ensure that we don't want to access indexes of the volume that are out of bound
    z_min, z_max = max(0, original_z_min), min(max_z, original_z_max)
    y_min, y_max = max(0, original_y_min), min(max_y, original_y_max)
    x_min, x_max = max(0, original_x_min), min(max_x, original_x_max)

    # Cut out the patch
    patch = volume[z_min:z_max, y_min:y_max, x_min:x_max]


    ## this way has the problem that the nodule will not be in the center anymore: 
    # Add padding to the patch if it falls (partly) out of the volume 
    # Made with the help of Gemini
    # if patch.shape != patch_size:
    #     pad_z = max(0, dz - patch.shape[0])
    #     pad_y = max(0, dy - patch.shape[1])
    #     pad_x = max(0, dx - patch.shape[2])
    #     # np.pad: ((before_z, after_z), (before_y, after_y), (before_x, after_x))
    #     # mode 'edge' repeats the edge value
    #     patch = np.pad(patch, ((0, pad_z), (0, pad_y), (0, pad_x)), mode='edge')

    ## Better:
    # in this way the nodule will be at the center
    if patch.shape != patch_size:
        patch = np.pad(patch, ((pad_before_z, pad_after_z), (pad_before_y, pad_after_y), (pad_before_x, pad_after_x)), mode="edge")


    return patch

### Apply preprocessing on all scans and convert to .npy files
We can preprocess all the scans upfront and save it to .npy files. Therefore we minimize the overhead during the training loop of processing and cutting out the patches.

In [ ]:
metadata_list = []
## Can be used later to get the path to the patch and label and let the model evaluate this patch. We can then compare it with the label of the row and backpropagate.
pos_count = 0
neg_count = 0
scans = pl.query(pl.Scan).all() 
for scan in tqdm(scans):
    patient_id = scan.patient_id
    processed_volume, zoom_factors = process_scan(scan)
    nodules = scan.cluster_annotations() ## Get all the annotations per nodule. Arrays is like: [[Nodule1_ann1, Nodule1_ann2,..],...[Nodule5_ann1, Nodule5_ann2?...],...]
    
    # Nodule patches (so both neg and pos patches)
    for i, nodule in enumerate(nodules): # so we extract [Nodule1_ann1, Nodule1_ann2,..] and then [Nodule2_ann1, Nodule2_ann2,..] ...

        raw_centroids = [ann.centroid for ann in nodule]
        mean_raw_center = np.mean(raw_centroids, axis=0) # (y, x, z), axis = 0 because we want to get one mean centroid. It calculates the mean of each row resulting in one coordinate instead of calculating the mean value over the row values (axis=1)

        ## Centroid is still in (X,Y,Z) format so needs to be converted to (Z,Y,X) and resampled with zoom_factors
        resampled_center = (
            mean_raw_center[2] * zoom_factors[2], # Z (Depth)
            mean_raw_center[0] * zoom_factors[0], # Y (Height)
            mean_raw_center[1] * zoom_factors[1]  # X (Width)
        )
        
        # Get mean malignancy of this nodule over all the opinions of the doctors
        mean_malignancy = np.mean([ann.malignancy for ann in nodule])
            
        label = 1 if mean_malignancy >= 3.0 else 0
        ## TODO: or we can skip patches that have a mean of 3.0 (now they are conceived as a bad nodule)
        
        # Get the patch
        patch = extract_patch(processed_volume, resampled_center, patch_size=(36, 48, 48))
        
        # Save in a Numpy file
        if label == 1: 
            patch_path = f"data/patches/{patient_id}_pos_{i}.npy"
            pos_count+=1
            np.save(patch_path, patch)
        else:
            patch_path = f"data/patches/{patient_id}_neg_{i}.npy"
            neg_count+=1
            np.save(patch_path, patch)

        metadata_list.append({
            "patient_id": patient_id,
            "series_instance_uid": scan.series_instance_uid, ## Unique identifier of a CT-scan. Needed if we want to visualise later the models prediction on the original 3D CT scan, with this we can extract the scan where this patch belongs to
            "patch_path": patch_path,
            "label": label,
            "mean_malignancy": mean_malignancy,
            "raw_center_z": mean_raw_center[2],
            "raw_center_y": mean_raw_center[0],
            "raw_center_x": mean_raw_center[1],
            "resampled_center_z": resampled_center[0],
            "resampled_center_y": resampled_center[1],
            "resampled_center_x": resampled_center[2]
        })
    

# Save metadata to CSV
df_metadata = pd.DataFrame(metadata_list)
df_metadata.to_csv("metadata.csv", index=False)

print("Total patches", (neg_count+pos_count))
print("Number of positive patches", (pos_count))
print("Number of negative patches", (neg_count))

  0%|          | 0/1018 [00:00<?, ?it/s]

RuntimeError: Couldn't find DICOM files for Scan(id=2,patient_id=LIDC-IDRI-0069) in C:\LIDC\lidc_idri\LIDC-IDRI-0069

# _Tests on data_

### _*1:  Is the voxel size different?_

In [23]:
scans = pl.query(pl.Scan).all()
for i in tqdm(range(5)):
    scan = scans[i]

    print("\nPatient:", scan.patient_id)

    volume = scan.to_volume(verbose=False)
    print("Volume shape:", volume.shape)
    print("Pixel spacing:", scan.pixel_spacing)
    print("Slice thickness:", scan.slice_thickness)


  0%|          | 0/5 [00:00<?, ?it/s]


Patient: LIDC-IDRI-0078
Volume shape: (512, 512, 87)
Pixel spacing: 0.65
Slice thickness: 3.0

Patient: LIDC-IDRI-0069
Volume shape: (512, 512, 136)
Pixel spacing: 0.741
Slice thickness: 2.0

Patient: LIDC-IDRI-0079
Volume shape: (512, 512, 113)
Pixel spacing: 0.698
Slice thickness: 3.0

Patient: LIDC-IDRI-0101
Volume shape: (512, 512, 150)
Pixel spacing: 0.626953125
Slice thickness: 3.0

Patient: LIDC-IDRI-0110
Volume shape: (512, 512, 125)
Pixel spacing: 0.73046875
Slice thickness: 3.0


Conclusion: yes voxel size differs. Likely because of the different origins (different scanners from different hospitals). 

Clear image to illustrate a voxel.and it's sice: https://pmc.ncbi.nlm.nih.gov/articles/PMC11202317/#:~:text=Fig%2E%201,-%2E
A voxel is like a 2D pixel but then in 3D. It is characterized by a Size = (Δx, Δy, Δz). Where:
* Δx = pixel spacing in the horizontal direction
* Δy = pixel spacing in the vertical direction
* Δz = slice thickness (or slice spacing, depending on the image metadata)

To ensure consistent spatial dimensions across all CT scans, we resample the volumes to a uniform voxel spacing of 1 × 1 × 1 mm³.
This allows the AI model to process anatomical structures at a consistent physical scale, reducing variations caused by differences in voxel spacing between scans.